# 03 — Feature Engineering as Logged Experiments

**Goal:** test the feature hypotheses raised by the EDA, one at a time, against the notebook-02 baseline (OOF balanced accuracy **0.94894**), keeping only what clears the noise floor.

## The experiment discipline

From here on, modeling is science, not vibes. Every idea follows the same contract, recorded in `experiments.md`:

1. **State the hypothesis first** — *why* should this help, in terms of something we observed in the EDA? An experiment without a mechanism is a lottery ticket.
2. **Change one thing.** Same folds, same model, same seed as the baseline — the delta then measures the change and nothing else.
3. **Keep only if ΔCV ≥ +0.0005.** Notebook 02 measured fold spread ≈ 0.0011–0.0017, so the OOF score's standard error is ≈ 0.0005. A gain below that is indistinguishable from luck, and a pile of "kept" lucky features is how you overfit CV itself. The threshold also buys *parsimony*: every kept feature is one more thing every later experiment carries.
4. **Drops are results too.** A falsified hypothesis about this dataset is knowledge; it's logged with the same care.

**Why not test all ideas at once?** Bundled changes confound: if the bundle gains +0.001, which part earned it? And if one part gains +0.002 while another loses −0.001, the bundle hides a keeper *and* ships a parasite. Isolate first, combine after (§8 checks the combination, because kept features can interact).

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

TARGET = 'health_condition'
NUM = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake']
CAT = ['diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender']
FEATURES = NUM + CAT
SEED = 42
CLASSES = ['at-risk', 'fit', 'unhealthy']
y = train[TARGET].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(train[FEATURES], y))

# Contract check: identical folds to notebook 02 (which saved fold_id per row).
fold_id = np.load('../artifacts/fold_id.npy')
for k, (tr, va) in enumerate(FOLDS):
    assert (fold_id[va] == k).all(), f"fold {k} mismatch vs notebook 02!"
print("fold assignment identical to notebook 02 ✓")

BASE_CV = 0.94894  # notebook 02 weighted baseline, same folds/model
KEEP_THRESHOLD = 0.0005

fold assignment identical to notebook 02 ✓


## Harness (copied from notebook 02) + a transform pipeline

Same `run_cv`/`make_submission` as notebook 02, plus one new idea: each experiment is a **transform function** `f(df) -> df` applied identically to train and test. This makes experiments composable — §8 can combine the winners by just chaining their transforms — and makes "what exactly did this experiment change?" answerable by reading one function.

In [2]:
def to_lgbm_frame(df_train, df_test):
    """category-dtype the non-numeric columns, aligning test categories to train's."""
    Xtr, Xte = df_train.copy(), df_test.copy()
    for c in Xtr.columns:
        if not pd.api.types.is_numeric_dtype(Xtr[c]):
            Xtr[c] = Xtr[c].astype('category')
            Xte[c] = Xte[c].astype(pd.CategoricalDtype(Xtr[c].cat.categories))
    return Xtr, Xte


def lgbm_weighted():
    return lgb.LGBMClassifier(
        objective='multiclass', num_class=3,
        n_estimators=1000, learning_rate=0.05,
        class_weight='balanced',
        random_state=SEED, n_jobs=-1, verbose=-1)


def run_cv(X, X_test, label=''):
    oof = np.zeros((len(X), len(CLASSES)))
    test_probs = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for k, (tr, va) in enumerate(FOLDS):
        model = lgbm_weighted()
        model.fit(X.iloc[tr], y[tr],
                  eval_set=[(X.iloc[va], y[va])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oof[va] = model.predict_proba(X.iloc[va])
        test_probs += model.predict_proba(X_test) / len(FOLDS)
        fold_scores.append(balanced_accuracy_score(y[va], oof[va].argmax(1)))
    cv = balanced_accuracy_score(y, oof.argmax(1))
    print(f"{label}: OOF {cv:.5f}  (Δ {cv - BASE_CV:+.5f})  fold std {np.std(fold_scores):.5f}")
    return cv, oof, test_probs


def make_submission(test_probs, name):
    labels = [CLASSES[i] for i in test_probs.argmax(1)]
    sub = pd.DataFrame({'id': test['id'], TARGET: labels})
    path = f'../submissions/{name}.csv'
    sub.to_csv(path, index=False)
    print(f"wrote {path}")
    return path


def run_experiment(name, transform):
    """Apply transform to raw train/test features, run CV, log the delta."""
    Xtr, Xte = to_lgbm_frame(transform(train[FEATURES].copy()),
                             transform(test[FEATURES].copy()))
    cv, oof, testp = run_cv(Xtr, Xte, label=name)
    RESULTS[name] = {'cv': cv, 'transform': transform, 'testp': testp}
    return cv

RESULTS = {}

## E1 — Missingness indicators

**Hypothesis (from EDA §5):** `bmi` missingness shifts the target distribution hard (unhealthy 8.4%→2.9%), `heart_rate` mildly. LightGBM already routes NaNs somewhere at each split, but that signal is *entangled* with the split's main threshold — the tree can't send "missing" one way independently of the value comparison at every node. Explicit 0/1 flags give the booster a clean handle, plus a row-level missing count (rows with many gaps may behave differently).

**Prediction:** small positive gain, driven mostly by `bmi_missing`. Cost: +14 cheap features.

In [3]:
def add_missing_flags(df):
    for c in FEATURES:
        df[c + '_miss'] = df[c].isna().astype('int8')
    df['n_missing'] = df[FEATURES].isna().sum(axis=1).astype('int8')
    return df

run_experiment('E1 missing flags', add_missing_flags)

E1 missing flags: OOF 0.94912  (Δ +0.00018)  fold std 0.00132


0.9491208426440833

**Verdict: Δ +0.00018 — below the bar, dropped.** Direction right, magnitude tiny. Interpretation: LightGBM's native NaN routing was already extracting most of the `bmi`-missingness signal on its own; the explicit flags added a handle the model barely needed. Worth remembering the *asymmetry*: flags cost nothing and would matter a lot for models without native NaN support.

## E2 — Imputation A/B

**Hypothesis (from EDA §5, stated as the *null* we expect to confirm):** filling numeric NaNs with the median and categorical NaNs with the mode **destroys** information — it silently asserts "missing ≈ typical," which EDA showed is false for `bmi`. For a GBM that handles NaN natively, imputation is not a neutral convenience; it should *hurt*.

We run it twice: **(a)** imputation alone — the naive-pipeline version; **(b)** imputation *plus* E1's flags — does keeping an explicit record of what was imputed rescue it?

**Why bother testing something we expect to fail?** Because "GBMs prefer NaN passthrough" is folklore we're converting into a measured fact about *this* dataset — and because imputation becomes unavoidable if we later add models that can't eat NaN (linear, NN). We need to know its price.

In [4]:
def impute_only(df):
    for c in NUM:
        df[c] = df[c].fillna(train[c].median())
    for c in CAT:
        df[c] = df[c].fillna(train[c].mode()[0])
    return df

def impute_plus_flags(df):
    df = add_missing_flags(df)
    return impute_only(df)

run_experiment('E2a impute median/mode', impute_only)
run_experiment('E2b impute + flags', impute_plus_flags)

E2a impute median/mode: OOF 0.90776  (Δ -0.04118)  fold std 0.00102


E2b impute + flags: OOF 0.94900  (Δ +0.00006)  fold std 0.00107


0.9490016888713962

*(Aside on leakage: strictly, imputation statistics should be computed inside each training fold. With 690k rows, a global median is numerically identical to any fold's median, so we take the shortcut — but note that for small data or target-dependent statistics (target encoding!), in-fold fitting is not optional. That's exactly why we're not target-encoding here: with 3-level categoricals, native handling already gives the tree every possible split, and target encoding would add leakage machinery for zero expressiveness gain.)*

**Verdicts: E2a Δ −0.04118 — catastrophic. E2b Δ +0.00006 — rescued to baseline.**

The most instructive result of the milestone. Median/mode imputation didn't just fail to help — it **destroyed 4 points of balanced accuracy**, exactly as the EDA predicted: filling `bmi`'s NaNs with the median asserts "missing ≈ typical," obliterating the strongest missingness signal in the data (unhealthy 8.4%→2.9% when missing) and polluting the feature's distribution besides. Adding flags (E2b) recovers baseline almost exactly: with the information preserved *somewhere*, the model finds it again.

**The transferable rule:** imputation is not a neutral preprocessing convenience — it's an information-destroying assertion. If a downstream model forces you to impute (linear, NN), always pair it with missingness indicators. Here, NaN passthrough remains champion.

## E3 — Ordinal encoding of the ordered categoricals

**Hypothesis (from EDA conclusion #5):** `stress_level`, `sleep_quality`, `physical_activity_level` have natural orders. Encoding them as ordered integers (NaN preserved) lets a *single* threshold express "at least medium stress."

**Honest counter-argument:** with only 3 levels, LightGBM's native categorical handling can already isolate any subset of levels in one split — ordinal adds no expressiveness here. The real case for ordinal encoding is high-cardinality ordered features (dozens of levels), where category-set search gets noisy. **Prediction: ≈ no change.** We test it anyway because the *combination* (ordinal + interactions in E4 built on these codes) is where order might pay.

In [5]:
ORD_MAPS = {
    'stress_level': {'low': 0, 'medium': 1, 'high': 2},
    'sleep_quality': {'poor': 0, 'average': 1, 'good': 2},
    'physical_activity_level': {'sedentary': 0, 'moderate': 1, 'active': 2},
}

def ordinal_encode(df):
    for c, m in ORD_MAPS.items():
        df[c] = df[c].map(m).astype('float32')  # NaN preserved
    return df

run_experiment('E3 ordinal encode', ordinal_encode)

E3 ordinal encode: OOF 0.94895  (Δ +0.00001)  fold std 0.00141


0.9489488745268616

**Verdict: Δ +0.00001 — nothing, exactly as predicted.** With 3 levels, native categorical handling already reaches every split ordinal encoding can express. Logged as confirmed-null: the value of this experiment is that nobody on this project ever needs to wonder about it again.

## E4 — Domain-derived interactions

**Hypothesis (from EDA conclusion #3):** the minority classes are *conditionally* gated — `fit` ≈ active + low stress + high steps + long sleep; `unhealthy` ≈ high stress + short sleep. Trees approximate such conjunctions with sequential splits, but each split is axis-aligned; explicit ratio/product features put the joint quantity on a single axis where one split suffices, freeing depth for other structure.

Features (each traceable to an EDA observation):
- `steps_per_ex_min` — intensity: same steps in less time separates active from padded exercise.
- `cal_per_step` — movement efficiency; calorie signal was weak alone, may only matter *relative to* activity.
- `sleep_x_quality` — duration × quality(ordinal): 8 poor hours ≠ 8 good hours.
- `stress_x_sleep` — stress(ordinal) × sleep duration: the two strongest features, whose *combination* defines `unhealthy` in the EDA.
- `activity_x_steps` — activity(ordinal) × steps: the `fit` gate.

**Prediction:** the most likely experiment to clear the bar, but modestly — the booster already approximates these with its 1000 trees.

In [6]:
def add_interactions(df):
    ords = {c: df[c].map(m) for c, m in ORD_MAPS.items()}
    df['steps_per_ex_min'] = df['step_count'] / (df['exercise_duration'] + 1)
    df['cal_per_step'] = df['calorie_expenditure'] / (df['step_count'] + 1)
    df['sleep_x_quality'] = df['sleep_duration'] * ords['sleep_quality']
    df['stress_x_sleep'] = ords['stress_level'] * df['sleep_duration']
    df['activity_x_steps'] = ords['physical_activity_level'] * df['step_count']
    return df

run_experiment('E4 interactions', add_interactions)

E4 interactions: OOF 0.94916  (Δ +0.00022)  fold std 0.00123


0.949158916595978

**Verdict: Δ +0.00022 — best of the ladder, still below the bar.** The mechanism is real (the booster's importances even like the new features) but 1000 trees of depth-wise splitting were already approximating these conjunctions well enough that putting them on a single axis bought little. Explicit interactions pay when trees are shallow, data is small, or interactions are deep — none of which is true here.

## §8 — Consolidation

Combine every experiment that individually cleared the keep-rule, then re-run CV on the union. **Why re-run instead of adding the deltas?** Features interact: two experiments can each earn +0.0008 by explaining the *same* rows (combined gain ≈ one alone), or unlock each other (combined > sum). The combined number is the only one that counts.

If nothing cleared the bar, the baseline feature set *is* the consolidated set — also a result: it would say this dataset's signal is already fully exposed to the booster, and the remaining points must come from the model side (notebook 04).

In [7]:
kept = {n: r for n, r in RESULTS.items() if r['cv'] - BASE_CV >= KEEP_THRESHOLD}
print("kept experiments:", list(kept) or "none")

def build_features(df):
    """All transformations that survived Milestone 3. Applied to raw features."""
    for r in kept.values():
        df = r['transform'](df)
    return df

if kept:
    cv, oof, testp = run_cv(*to_lgbm_frame(build_features(train[FEATURES].copy()),
                                           build_features(test[FEATURES].copy())),
                            label='combined')
    np.save('../artifacts/oof_nb03_combined.npy', oof)
    np.save('../artifacts/testp_nb03_combined.npy', testp)
else:
    cv, testp = BASE_CV, RESULTS[max(RESULTS, key=lambda n: RESULTS[n]['cv'])]['testp']
    print("no keepers; carrying baseline feature set forward")

kept experiments: none
no keepers; carrying baseline feature set forward


## Submission

In [8]:
best_name, best = max(RESULTS.items(), key=lambda kv: kv[1]['cv'])
if kept and cv >= best['cv']:
    print(f"submitting combined (CV {cv:.5f})")
    make_submission(testp, '03-features')
else:
    print(f"submitting best single experiment: {best_name} (CV {best['cv']:.5f})")
    make_submission(best['testp'], '03-features')

submitting best single experiment: E4 interactions (CV 0.94916)


wrote ../submissions/03-features.csv


### Result

| submission | CV | public LB | gap (CV − LB) |
|---|---|---|---|
| nb02 baseline | 0.94894 | 0.94667 | +0.0023 |
| nb03 E4 interactions | 0.94916 | **0.94942** | **−0.0003** |

Two things worth staring at:

1. **The gap flipped sign.** CV moved +0.0002 but the LB moved +0.0028. Both submissions can't be "the true gap" — the public LB is a sample with its own noise, amplified for balanced accuracy because minority-class recall there rests on few rows. Lesson: the CV↔LB gap is a *distribution*, not a constant. We now estimate it as roughly ±0.002-0.003 around zero. Decisions stay CV-driven.
2. **Don't celebrate LB moves your CV didn't predict.** The +0.0028 LB jump overstates a +0.0002 improvement. The reverse error (panicking over an unlucky LB draw) wastes submissions chasing ghosts.

### Milestone conclusion

**The baseline feature set is already sufficient — the booster sees everything.** Five hypotheses, five clean answers, zero keepers: this synthetic dataset's signal is fully exposed to a GBM without our help. That is not a failed milestone; it's a redirect with evidence: the remaining ~0.005 to the leaderboard top must come from the *model side* — undertrained baseline (every fold still hits the 1000-tree cap), hyperparameters, per-class decision thresholds, and ensembling. That is exactly notebook 04's agenda.